# ML - Aprendizaje No Supervisado
## K-Means con Dataset Económico — 3 Variables
### Dataset: Big Mac Index (The Economist)

El aprendizaje no supervisado busca **descubrir patrones ocultos** en los datos sin etiquetas. El algoritmo K-Means agrupa países con comportamiento económico similar sin que nadie le diga de antemano cuáles van juntos.

## Descripción del Dataset: Big Mac Index

### ¿De qué se trata?

El **Big Mac Index** fue creado por la revista **The Economist** en 1986 como una forma informal pero efectiva de comparar el **poder adquisitivo** entre países. Se basa en la teoría de la *Paridad del Poder Adquisitivo (PPA)*: en el largo plazo, los tipos de cambio deberían ajustarse para que un mismo producto cueste lo mismo en cualquier país del mundo.

La hamburguesa Big Mac fue elegida porque se produce de forma estandarizada en más de 100 países, usando ingredientes locales, lo que la convierte en un indicador práctico del costo de vida.

### ¿Qué contiene?

| Columna | Tipo | Descripción |
|---|---|---|
| **name** | Categórico | País o región |
| **local_price** | Numérico | Precio del Big Mac en moneda local |
| **dollar_ex** | Numérico | Tipo de cambio frente al dólar |
| **dollar_price** | Numérico | Precio del Big Mac convertido a dólares |
| **gdp_dollar** | Numérico | PIB per cápita en dólares |
| **usd_raw** | Numérico | % de sub/sobrevaluación de la moneda (crudo) |
| **adj_price** | Numérico | Precio ajustado por PIB |

### ¿Para qué sirve este análisis?

Aplicando K-Means sobre variables económicas reales podemos **agrupar países** según su nivel de precios, tipo de cambio y PIB. Esto permite identificar:
- Países con monedas **sobrevaluadas** (precio en dólares alto)
- Países con monedas **subvaluadas** (precio en dólares bajo)
- Países con **alto PIB y alto precio** vs **bajo PIB y bajo precio**

### Columnas utilizadas en este notebook (3 variables)
- **`dollar_price`** → precio del Big Mac en dólares
- **`dollar_ex`** → tipo de cambio frente al dólar
- **`usd_raw`** → porcentaje de sub/sobrevaluación de la moneda

## 1. Carga del Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Dataset Big Mac Index - The Economist (descarga directa)
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-12-22/big-mac.csv"
df_raw = pd.read_csv(url)

print("Primeras filas:")
print(df_raw.head())
print(f"\nDimensiones: {df_raw.shape}")
print(f"\nColumnas: {list(df_raw.columns)}")

## 2. Limpieza y Selección de 3 Variables

In [ ]:
# Tomamos el registro más reciente por país y eliminamos valores nulos
df = df_raw.sort_values('date').groupby('name').last().reset_index()
df = df[['name', 'dollar_price', 'dollar_ex', 'usd_raw']].dropna()

print(f"Países disponibles: {len(df)}")
print(df.head(15))
print("\nEstadísticas descriptivas:")
print(df.describe())

In [ ]:
# Seleccionamos las 3 columnas numéricas para el clustering
X = df[['dollar_price', 'dollar_ex', 'usd_raw']].values

print("Shape de X:", X.shape)
print("Primeros 5 valores:\n", X[:5])

## 3. Normalización de los Datos

Como las 3 variables tienen escalas muy distintas (el tipo de cambio puede ser miles, mientras que usd_raw va de -1 a 1), es importante **normalizar** antes de aplicar K-Means para que ninguna variable domine por su magnitud.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Datos normalizados (media~0, std~1):")
print(pd.DataFrame(X_scaled, columns=['dollar_price', 'dollar_ex', 'usd_raw']).describe().round(2))

## 4. Visualización Inicial (sin clusters) en 3D

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(X_scaled[:, 0], X_scaled[:, 1], X_scaled[:, 2],
           c='black', s=20, alpha=0.7)

ax.set_xlabel('Precio en USD (norm.)')
ax.set_ylabel('Tipo de Cambio (norm.)')
ax.set_zlabel('Sobrevaluación USD (norm.)')
ax.set_title('Big Mac Index — Países sin clustering (3 variables)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Entrenamiento K-Means (k=4)

In [ ]:
from sklearn.cluster import KMeans

k = 4  # 4 grupos de países según comportamiento económico
kmeans = KMeans(n_clusters=k, random_state=42)
y_pred = kmeans.fit_predict(X_scaled)

df['cluster'] = y_pred
print("Centroides encontrados (escala normalizada):")
print(pd.DataFrame(kmeans.cluster_centers_,
                   columns=['dollar_price', 'dollar_ex', 'usd_raw']).round(3))

## 6. Visualización de Clusters en 3D

In [ ]:
colores = ['#E74C3C', '#2ECC71', '#3498DB', '#F39C12']

fig = plt.figure(figsize=(11, 7))
ax = fig.add_subplot(111, projection='3d')

for i in range(k):
    mask = y_pred == i
    ax.scatter(X_scaled[mask, 0], X_scaled[mask, 1], X_scaled[mask, 2],
               c=colores[i], s=40, label=f'Cluster {i}', alpha=0.8)

# Centroides
centroids = kmeans.cluster_centers_
ax.scatter(centroids[:, 0], centroids[:, 1], centroids[:, 2],
           c='black', s=300, marker='X', zorder=10, label='Centroides')

ax.set_xlabel('Precio USD (norm.)')
ax.set_ylabel('Tipo de Cambio (norm.)')
ax.set_zlabel('Sobrevaluación (norm.)')
ax.set_title('K-Means — Big Mac Index (3 variables)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 7. ¿Qué países quedaron en cada cluster?

In [ ]:
for i in range(k):
    paises = df[df['cluster'] == i]['name'].tolist()
    avg_price = df[df['cluster'] == i]['dollar_price'].mean()
    avg_eval = df[df['cluster'] == i]['usd_raw'].mean()
    print(f"\n🔵 Cluster {i} — Precio promedio: ${avg_price:.2f} | Sobrevaluación promedio: {avg_eval:.2%}")
    print(f"   Países: {', '.join(paises)}")